# OpenPlaque — Cross-Rotation Consensus Coronary Tracking

This notebook uses a property specific to the Siemens curved coronary reformat stack: each frame is an alternate rotation around the same vessel. The target coronary should remain relatively stable in CPR pixel coordinates while chambers, bones, and other anatomy move with rotation.

The workflow therefore builds **persistent coronary evidence across every rotation first**, tracks on that consensus image, and only then selects the individual CPR rotation that displays the fixed path best. Plaque is **not used to generate or choose the path**; it is overlaid afterward for visualization.

Every persistent component has a Boolean reuse flag. All default to `True`: valid cache → reuse; missing/invalid cache → recompute and cache; `False` → force recomputation and replace the cache. Canonical TPV is unchanged. CPR plaque views and source-volume PCAT are not spatially co-registered. Research use only.


## Step 1 — Mount Google Drive

In [ ]:
# FIRST EXECUTABLE CELL — Drive mount must remain first.
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache / reuse controls

In [ ]:
# Default behavior: reuse every valid cache.
REUSE_SERIES_SELECTION = True
REUSE_PLAQUE_MASKS = True
REUSE_CONSENSUS_EVIDENCE = True
REUSE_TRACKING = True
REUSE_CANDIDATE_FIGURE = True
REUSE_ROADMAPS = True
REUSE_PCAT_FIGURES = True
REUSE_DASHBOARD = True
REUSE_REPORT_PACKAGE = True


## Step 3 — Install this fresh branch

In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch cross-rotation-consensus-tracking-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!pip -q install -r /content/OpenPlaque/requirements-colab.txt

import sys
sys.path.insert(0, '/content/OpenPlaque/src')
import pandas as pd
from IPython.display import display, Image
from openplaque.consensus_tracking_workflow import ConsensusTrackingWorkflow

print('Cross-rotation consensus tracker ready.')


## Step 4 — Initialize and inspect the cache plan

The table shows which persistent components are currently available and whether this run plans to reuse or recompute them.


In [ ]:
REUSE = {
    'series_selection': REUSE_SERIES_SELECTION,
    'plaque_masks': REUSE_PLAQUE_MASKS,
    'consensus_evidence': REUSE_CONSENSUS_EVIDENCE,
    'tracking': REUSE_TRACKING,
    'candidate_figure': REUSE_CANDIDATE_FIGURE,
    'roadmaps': REUSE_ROADMAPS,
    'pcat_figures': REUSE_PCAT_FIGURES,
    'dashboard': REUSE_DASHBOARD,
    'report_package': REUSE_REPORT_PACKAGE,
}
wf = ConsensusTrackingWorkflow('/content/drive/MyDrive/OpenPlaque', REUSE)
display(wf.cache_status())


## Step 5 — Prepare DICOM and coronary CPR series

In [ ]:
series_map = wf.prepare_inputs()
print('Series:', series_map)


## Step 6 — Load canonical plaque masks

Existing validated plaque masks are reused if geometry and canonical TPV agree. nnU-Net runs only on a genuine cache miss or when `REUSE_PLAQUE_MASKS=False`. These masks are **not** used to generate the consensus path.


In [ ]:
cache_qc = wf.load_plaque_masks()
display(cache_qc)


## Step 7 — Build cross-rotation consensus evidence

For every CPR rotation, the code computes coronary-scale vesselness/local-contrast evidence. It then combines the lower quartile, median, upper quantile, and persistence fraction across rotations. Structures that are bright in only a few rotations are suppressed.


In [ ]:
consensus = wf.build_consensus_evidence()
for vessel in ('LAD','RCA','LCX'):
    c = consensus[vessel]['consensus']
    s = consensus[vessel]['support']
    print(vessel, 'consensus max=', float(c.max()), 'mean support=', float(s[c>0].mean()) if (c>0).any() else 0)


## Step 8 — Track the persistent coronary path and choose the best rotation

Tracking is performed on the consensus image, not independently on each rotation. The selected path is fixed first; the best individual rotation is chosen afterward using lumen HU plausibility and local path-versus-background contrast. Plaque does not participate in either decision.


In [ ]:
tracking_qc = wf.track_consensus_paths()
display(tracking_qc)


## Step 9 — Review consensus candidates

Each row shows the top three persistent paths for one artery on the same cross-rotation consensus image. The first column is the actual selected path.


In [ ]:
candidate_png = wf.plot_candidates()
print('Saved:', candidate_png)
display(Image(filename=str(candidate_png)))


## Step 10 — Create best-rotation and straightened roadmaps

The left column shows the fixed consensus path on its best CPR rotation. Canonical plaque is overlaid only after tracking. The right column straightens the same path.


In [ ]:
roadmap_png = wf.plot_roadmaps()
print('Saved:', roadmap_png)
display(Image(filename=str(roadmap_png)))
if wf.along_df is not None:
    display(wf.along_df.head(40))


## Step 11 — RCA OpenPlaque PCAT Attenuation figures

The default reuses the validated PCAT visualization. Set `REUSE_PCAT_FIGURES=False` to regenerate it from the frozen RCA centerline/radius/aorta inputs.


In [ ]:
pcat_files = wf.pcat_figures()
print(pcat_files)
display(Image(filename=str(pcat_files['cross_sections'])))
display(Image(filename=str(pcat_files['ribbon'])))


## Step 12 — Build the summary dashboard

In [ ]:
dashboard = wf.plot_dashboard()
print('Saved:', dashboard)
display(Image(filename=str(dashboard)))


## Step 13 — Package the report-back ZIP

The ZIP includes `cache_provenance.csv` as well as all QC tables and figures.


In [ ]:
zip_path = wf.package_report()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_CROSS_ROTATION_CONSENSUS_REPORT_BACK.zip')
print('\nCache provenance:')
display(pd.read_csv(wf.out / 'cache_provenance.csv'))
